In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
from tqdm import tqdm
import os
from urllib.parse import urljoin

# Set request headers
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Referer': 'https://pubmed.ncbi.nlm.nih.gov/',
    'DNT': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
}

# Page URLs
BASE_URL = "https://pubmed.ncbi.nlm.nih.gov"
SEARCH_URL = "https://pubmed.ncbi.nlm.nih.gov/?term=bee+pollination&sort=date"  # Current example: keyword is "bee pollination" and sorting is set to "most recent". Also used: 2.keyword is "bee pollination" and sorting is set to "best match" 3.keyword is "honeybee" and sorting is set to "most recent" 4.keyword is "honeybee" and sorting is set to "best match"


# Output file
CSV_FILE = "pubmed_articles.csv"

# Variables
all_articles = []
article_count = 0
start_page = 1
max_pages = 2000  # Maximum total number of pages to crawl

# Check whether a saved file already exists; if so, resume from the last interruption
if os.path.exists(CSV_FILE):
    try:
        existing_data = pd.read_csv(CSV_FILE)
        article_count = len(existing_data)
        start_page = (article_count // 10) + 1
        all_articles = existing_data.to_dict('records')
        print(f"Existing data detected. Resuming from page {start_page}; {article_count} records have already been collected.")
    except Exception as e:
        print(f"Error reading the existing file: {e}. Crawling will restart from the beginning.")


def get_full_abstract(article_url):
    """Get the full abstract from the article detail page."""
    try:
        # Random delay to avoid sending requests too frequently
        time.sleep(random.uniform(2, 5))

        response = requests.get(article_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Look for the abstract container - on PubMed, abstracts are usually inside divs with class="abstract-content"
        abstract_sections = soup.find_all('div', class_='abstract-content')

        if not abstract_sections:
            # Try other possible abstract containers
            abstract_sections = soup.find_all('div', class_='abstract')

        if abstract_sections:
            # Combine all abstract paragraphs
            full_abstract = "\n".join(
                section.get_text(strip=True)
                for section in abstract_sections
            )
            return full_abstract
        else:
            return "No full abstract available"

    except Exception as e:
        print(f"Failed to retrieve full abstract: {article_url} - {str(e)}")
        return "Abstract retrieval failed"


# Create a progress bar
with tqdm(total=max_pages * 10, initial=article_count, desc="Crawling progress") as pbar:
    # Iterate through all pages
    for page_num in range(start_page, max_pages + 1):
        # Build the page URL
        page_url = f"{SEARCH_URL}&page={page_num}"

        # Send request
        try:
            response = requests.get(page_url, headers=HEADERS)
            response.raise_for_status()  # Check whether the request succeeded
        except Exception as e:
            print(f"Request for page {page_num} failed: {e}")
            time.sleep(10)  # Wait 10 seconds before retrying
            continue

        # Parse the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all article containers
        articles = soup.find_all('div', class_='docsum-wrap')

        # If no articles are found, the page structure may have changed or the crawler may have reached the last page
        if not articles:
            print(f"No articles were found on page {page_num}. The page structure may have changed or the last page may have been reached.")
            break

        # Process each article
        for article in articles:
            try:
                # Extract title and link
                title_element = article.find('a', class_='docsum-title')
                if title_element:
                    title = title_element.get_text(strip=True)
                    link = urljoin(BASE_URL, title_element['href'])
                else:
                    title = "Untitled"
                    link = "No link"

                # Get the full abstract
                full_abstract = get_full_abstract(link) if link != "No link" else "No link"

                # Add to the list
                article_count += 1
                all_articles.append({
                    'Record No.': article_count,
                    'Title': title,
                    'Link': link,
                    'Full Abstract': full_abstract
                })

                # Notify the user
                print(f"Collected record {article_count}: {title[:50]}...")

                # Update the progress bar
                pbar.update(1)

                # Save every 5 records to reduce save frequency
                if article_count % 5 == 0:
                    df = pd.DataFrame(all_articles)
                    df.to_csv(CSV_FILE, index=False, encoding='utf-8-sig')
                    print(f"Saved {article_count} records to {CSV_FILE}")

            except Exception as e:
                print(f"Error while processing an article: {e}")
                continue

        # Wait a random 5–10 seconds to avoid sending requests too frequently
        wait_time = random.uniform(5, 10)
        time.sleep(wait_time)

        # Check whether there is a next page
        next_button = soup.find('button', class_='next-page-btn')
        if not next_button or 'disabled' in next_button.get('class', []):
            print("Reached the last page.")
            break

# Save the final results
if all_articles:
    df = pd.DataFrame(all_articles)
    df.to_csv(CSV_FILE, index=False, encoding='utf-8-sig')
    print(f"Crawling completed. A total of {len(all_articles)} records were collected, and the results have been saved to {CSV_FILE}.")
else:
    print("No data was collected.")


检测到已有数据，从第 437 页继续爬取，已爬取 4368 条记录


爬取进度:  19%|█▊        | 4368/23600 [00:00<?, ?it/s]

请求第 437 页失败: 403 Client Error: Forbidden for url: https://pubmed.ncbi.nlm.nih.gov/?term=bee+pollination&sort=date&page=437
请求第 438 页失败: 403 Client Error: Forbidden for url: https://pubmed.ncbi.nlm.nih.gov/?term=bee+pollination&sort=date&page=438
请求第 439 页失败: 403 Client Error: Forbidden for url: https://pubmed.ncbi.nlm.nih.gov/?term=bee+pollination&sort=date&page=439
